In [ ]:
# =============================================================================
# State-Space LTI System Analysis — Problem 090701 (Symbolic Solution)
# =============================================================================

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Math, HTML

# Disable scrolling for large outputs in Jupyter Notebook
display(HTML("<style>.output_scroll { max-height: none !important; }</style>"))

sp.init_printing(use_unicode=True)
z, n = sp.symbols('z n', complex=True)

print("=== State-Space System Response Analysis — Problem 090701 ===")
print()

# =============================================================================
# 1. System Matrices & Initial Conditions Definition
# =============================================================================
display(Math(r"\text{1. System Matrices and Parameters}"))

A = sp.Matrix([
    [0, 1],
    [-sp.Rational(1, 6), sp.Rational(5, 6)]
])

B = sp.Matrix([
    [0],
    [1]
])

C = sp.Matrix([[-1, 5]])
D = sp.Matrix([[0]])

q0 = sp.Matrix([2, 3])

display(Math(r"A = " + sp.latex(A)))
display(Math(r"B = " + sp.latex(B)))
display(Math(r"C = " + sp.latex(C)))
display(Math(r"D = " + sp.latex(D)))
display(Math(r"q[0] = " + sp.latex(q0)))

# =============================================================================
# 2. Zero-Input Component Calculation
# =============================================================================
display(Math(r"\text{2. Zero-Input State Vector Components}"))

I = sp.eye(2)
inv_resolvent_zi = (I - (z**(-1)) * A).inv()
Q_zi_z = inv_resolvent_zi * q0

display(Math(r"(I - z^{-1}A)^{-1}q[0] = " + sp.latex(Q_zi_z)))

q_zi_list = []
for i in range(2):
    expr = Q_zi_z[i]
    aux_expr = expr * z**(n - 1)
    denom = sp.denom(sp.together(expr))
    poles = sp.solve(sp.Eq(denom, 0), z)
    
    res_sum = 0
    for p in poles:
        r = sp.residue(aux_expr, z, p)
        res_sum += r
    
    q_zi_i = sp.factor(sp.simplify(res_sum))
    q_zi_list.append(q_zi_i)

q_zi = sp.Matrix(q_zi_list)
display(Math(r"q_{zi}[n] = " + sp.latex(q_zi)))

# =============================================================================
# 3. Zero-State Component Calculation (Input x[n] = u[n] => X(z) = z / (z - 1))
# =============================================================================
display(Math(r"\text{3. Zero-State State Vector Components}"))

X_z = z / (z - 1)
inv_resolvent_zs = (z * I - A).inv()
Q_zs_z = inv_resolvent_zs * B * X_z

display(Math(r"\mathcal{X}^+(z) = \frac{z}{z-1}"))
display(Math(r"(zI - A)^{-1}B \mathcal{X}^+(z) = " + sp.latex(Q_zs_z)))

q_zs_list = []
for i in range(2):
    expr = Q_zs_z[i]
    aux_expr = expr * z**(n - 1)
    denom = sp.denom(sp.together(expr))
    poles = sp.solve(sp.Eq(denom, 0), z)
    
    res_sum = 0
    for p in poles:
        r = sp.residue(aux_expr, z, p)
        res_sum += r
        
    q_zs_i = sp.factor(sp.simplify(res_sum))
    q_zs_list.append(q_zs_i)

q_zs = sp.Matrix(q_zs_list)
display(Math(r"q_{zs}[n] = " + sp.latex(q_zs)))

# =============================================================================
# 4. Total State Vector q[n]
# =============================================================================
display(Math(r"\text{4. Total State Vector } q[n]"))

q_total = sp.factor(sp.simplify(q_zi + q_zs))
display(Math(r"q[n] = q_{zi}[n] + q_{zs}[n] = " + sp.latex(q_total)))

# =============================================================================
# 5. System Output y[n] Calculation (Zero-Input + Zero-State Outputs)
# =============================================================================
display(Math(r"\text{5. System Output } y[n]"))

Y_zi_z = (C * inv_resolvent_zi * q0)[0]
aux_y_zi = Y_zi_z * z**(n - 1)
poles_y_zi = sp.solve(sp.Eq(sp.denom(sp.together(Y_zi_z)), 0), z)
y_zi = sp.factor(sp.simplify(sum(sp.residue(aux_y_zi, z, p) for p in poles_y_zi)))

Y_zs_z = ((C * inv_resolvent_zs * B + D) * X_z)[0]
aux_y_zs = Y_zs_z * z**(n - 1)
poles_y_zs = sp.solve(sp.Eq(sp.denom(sp.together(Y_zs_z)), 0), z)
y_zs = sp.factor(sp.simplify(sum(sp.residue(aux_y_zs, z, p) for p in poles_y_zs)))

y_total = sp.factor(sp.simplify(y_zi + y_zs))

display(Math(r"y_{zi}[n] = " + sp.latex(y_zi)))
display(Math(r"y_{zs}[n] = " + sp.latex(y_zs)))
display(Math(r"y[n] = y_{zi}[n] + y_{zs}[n] = \left(" + sp.latex(y_total) + r"\right)u[n]"))

# =============================================================================
# 6. Numerical Evaluation & Discrete Signal Visualization (Stem Plot)
# =============================================================================
display(Math(r"\text{6. Total Output Visualization}"))

def evaluate_symbolic(expr, n_values):
    return np.array([float(sp.re(sp.N(expr.subs(n, int(k))))) for k in n_values])

out = widgets.Output()

def plot_system_response(N=20):
    with out:
        out.clear_output(wait=True)
        n_vec = np.arange(0, N)
        y_vals = evaluate_symbolic(y_total, n_vec)

        fig, axes = plt.subplots(1, 1, figsize=(10, 4))

        markerline, stemlines, baseline = axes.stem(n_vec, y_vals, basefmt="k-")
        plt.setp(markerline, markersize=6, markerfacecolor='red', markeredgecolor='red')
        plt.setp(stemlines, linewidth=1.5, color='blue')

        axes.set_title(r"Total System Output y[n] (Discrete Signal)", fontsize=11)
        axes.set_xlabel(r"n (Time Index)")
        axes.set_ylabel(r"y[n]")
        axes.grid(True, linestyle='--', alpha=0.6)

        plt.tight_layout()
        plt.show()

plot_system_response()
display(out)